# Tutorial: Document Topic Classification Using scikit-learn

In this tutorial we will see how to:
- Load Dataset
- Build feature space using feature engineering techniques (Bag-of-words)
- Train a model classifier
- Evaluate the classifier
- Use the model on new documents

## Load Dataset

The dataset is called “Twenty Newsgroups”. It is a collection of approximately 20,000 newsgroup documents, partitioned evenly across 20 different newsgroups.
In order to get faster, we will work on a partial dataset with only 4 categories out of the 20 available in the dataset.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

twenty_train = fetch_20newsgroups(subset='train',
    categories=categories, shuffle=True, random_state=42)

twenty_test = fetch_20newsgroups(subset='test',
    categories=categories, shuffle=True, random_state=42)

### Print the target names of the dataset

In [ ]:
print(twenty_train.target_names)
print(twenty_test.target_names)

### Print the number of documents in the training and the testing dataset

In [ ]:
print("The number of training docs:", len(twenty_train.data)) 
print("The number of testing docs:", len(twenty_test.data)) 

### Print the first document and it's target

In [ ]:
print(twenty_train.data[0])
print("Document category:", twenty_train.target_names[twenty_train.target[0]])

## Build feature space using feature engineering techniques (Bag-of-words)

Text preprocessing, tokenizing and filtering of stopwords are all included in CountVectorizer, which builds a dictionary of features and transforms documents to feature vectors.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(twenty_train.data)

print(X_train_counts.shape)

### From occurrences to frequencies using TF-IDF

Occurrence count is a good start but there is an issue: longer documents will have higher average count values than shorter documents, even though they might talk about the same topics.

To avoid these potential discrepancies it suffices to divide the number of occurrences of each word in a document by the total number of words in the document: these new features are called tf for Term Frequencies.

Another refinement on top of tf is to downscale weights for words that occur in many documents in the corpus and are therefore less informative than those that occur only in a smaller portion of the corpus.

This downscaling is called tf–idf for “Term Frequency times Inverse Document Frequency”.

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)

X_train_tfidf.shape

## Train a model classifier
We train two classifiers one using CountVectorizer and the other using TfIdfVectorizer

In [ ]:
from sklearn.naive_bayes import MultinomialNB

clf_count = MultinomialNB().fit(X_train_counts, twenty_train.target)
clf_tfidf = MultinomialNB().fit(X_train_tfidf, twenty_train.target)

## Evaluate And Compare the two models

In [ ]:
import numpy as np

X_testing_counts = count_vect.transform(twenty_test.data)
X_testing_tfidf = tfidf_transformer.transform(X_testing_counts)

predicted_count = clf_count.predict(X_testing_counts)
accuracy_count = np.mean(predicted_count == twenty_test.target)

predicted_tfidf = clf_tfidf.predict(X_testing_tfidf)
accuracy_tfidf = np.mean(predicted_tfidf == twenty_test.target)

print("The accuracy of the model trained using CountVectorizer:", accuracy_count)
print("The accuracy of the model trained using TfIdfVectorizer:", accuracy_tfidf)

scikit-learn provides further utilities for more detailed performance analysis of the results

In [ ]:
from sklearn import metrics

print("##################################################################################")
print("Model trained using CountVectorizer")
print(metrics.classification_report(twenty_test.target, predicted_count,
    target_names=twenty_test.target_names))

print("##################################################################################")
print("Model trained using TfIdfVectorizer")
print(metrics.classification_report(twenty_test.target, predicted_tfidf,
    target_names=twenty_test.target_names))




## Use the model to predict for new documents

In [ ]:
docs_new = [
    """A graphics processing unit (GPU) is a specialized electronic circuit initially designed to accelerate computer graphics and image processing (either on a video card or embedded on motherboards, mobile phones, personal computers, workstations, and game consoles).""", 
    """Paracetamol (acetaminophen[a] or para-hydroxyacetanilide) is a non-opioid analgesic and antipyretic agent used to treat fever and mild to moderate pain. It is a widely used over the counter medication"""
]

X_new_counts = count_vect.transform(docs_new)
X_new_tfidf = tfidf_transformer.transform(X_new_counts)

predicted = clf_tfidf.predict(X_new_tfidf)

for doc, category in zip(docs_new, predicted):
    print('%r => \n predicition: %s\n' % (doc, twenty_train.target_names[category]))

## Easier way using scikit-learn Pipeline
In order to make the vectorizer => transformer => classifier easier to work with, scikit-learn provides a Pipeline class that behaves like a compound classifier

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier

text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier(loss='hinge', penalty='l2',
                         alpha=1e-3, random_state=42,
                         max_iter=5, tol=None)),
])

text_clf.fit(twenty_train.data, twenty_train.target)

predicted = text_clf.predict(twenty_test.data)

np.mean(predicted == twenty_test.target)

In [ ]:
docs_new = [
    """A graphics processing unit (GPU) is a specialized electronic circuit initially designed to accelerate computer graphics and image processing (either on a video card or embedded on motherboards, mobile phones, personal computers, workstations, and game consoles).""", 
    """Paracetamol (acetaminophen[a] or para-hydroxyacetanilide) is a non-opioid analgesic and antipyretic agent used to treat fever and mild to moderate pain. It is a widely used over the counter medication"""
]

predicted = text_clf.predict(docs_new)

for doc, category in zip(docs_new, predicted):
    print('%r => \n %s\n' % (doc, twenty_train.target_names[category]))